# 홈캠 얼굴 식별 + 사람 ByteTrack v1

정면 홈캠에서 YOLO 사람 검출과 ByteTrack을 실행하고, SCRFD·ArcFace 결과를 사람 트랙에 연결합니다. 이 노트북은 **1단계 홈캠 검증 전용**이며 CCTV 간 신원 인계는 구현하지 않습니다. `q`로 종료하면 개인정보가 없는 진단 JSON을 자동 저장합니다.

## 1. 환경과 모델 경로


In [5]:
from __future__ import annotations

import ctypes
import json
import os
import sys
import threading
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'deeplearning').is_dir() and (candidate / 'webapps').is_dir():
            return candidate
    raise RuntimeError('smart_office_monitoring 저장소 안에서 실행하세요.')

def load_env_file(path: Path) -> None:
    if not path.is_file():
        return
    for raw in path.read_text(encoding='utf-8-sig').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = (item.strip() for item in line.split('=', 1))
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {'\"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)

PROJECT_ROOT = find_project_root()
for env_path in (
    PROJECT_ROOT / 'deeplearning/training/.env.face',
    PROJECT_ROOT / 'webapps/fastapi/.env',
    PROJECT_ROOT / 'webapps/fastapi/.env.local',
    PROJECT_ROOT / 'deeplearning/training/.env',
    PROJECT_ROOT / 'deeplearning/training/.env.local',
):
    load_env_file(env_path)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_ROOT = PROJECT_ROOT / 'deeplearning/.models'
HOME_CAM_SOURCE = os.environ.get('HOME_CAM_SOURCE', 'rtsp').strip().lower()
HOME_CAM_RTSP_URL = os.environ.get('HOME_CAM_RTSP_URL', '').strip()
HOME_CAM_CAMERA_INDEX = int(os.environ.get('HOME_CAM_CAMERA_INDEX', '0'))
if HOME_CAM_SOURCE not in {'rtsp', 'webcam'}:
    raise ValueError('HOME_CAM_SOURCE는 rtsp 또는 webcam이어야 합니다.')
if HOME_CAM_SOURCE == 'rtsp' and not HOME_CAM_RTSP_URL.startswith('rtsp://'):
    raise ValueError('rtsp 모드에서는 HOME_CAM_RTSP_URL=rtsp://... 설정이 필요합니다.')
MONGODB_URI = os.environ.get('MONGODB_URI') or os.environ.get('DATABASE_URL', '')
MONGODB_DATABASE = os.environ.get('MONGODB_DATABASE') or os.environ.get('DATABASE_NAME', '')
COLLECTION_NAME = os.environ.get('FACE_EMBEDDING_COLLECTION', 'face_embeddings')
DETECTOR_PATH = Path(os.environ.get('FACE_DETECTION_MODEL_PATH') or MODEL_ROOT / 'scrfd/scrfd_10g_bnkps.onnx').resolve()
RECOGNIZER_PATH = Path(os.environ.get('FACE_RECOGNITION_MODEL_PATH') or MODEL_ROOT / 'buffalo_l/w600k_r50.onnx').resolve()
THRESHOLD_FILE = Path(os.environ['OPEN_SET_THRESHOLD_FILE']).resolve()
PERSON_MODEL_PATH = os.environ.get('HOME_CAM_PERSON_MODEL_PATH', 'yolo11m.pt')
PERSON_CONFIDENCE = float(os.environ.get('HOME_CAM_PERSON_CONFIDENCE', '0.25'))
TRACKER_CONFIG = os.environ.get('HOME_CAM_TRACKER_CONFIG', 'bytetrack.yaml')
FACE_COVERAGE_THRESHOLD = float(os.environ.get('HOME_CAM_FACE_COVERAGE_THRESHOLD', '0.80'))
FACE_INTERVAL = int(os.environ.get('FACE_RECOGNITION_INTERVAL', '6'))
DIAGNOSTIC_OUTPUT_DIR = Path(os.environ.get('HOME_CAM_DIAGNOSTIC_OUTPUT_DIR') or PROJECT_ROOT / 'deeplearning/training/runs/homecam_tracking')
for required_path in (DETECTOR_PATH, RECOGNIZER_PATH, THRESHOLD_FILE):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)
if not MONGODB_URI or not MONGODB_DATABASE:
    raise RuntimeError('MongoDB 연결 설정이 필요합니다.')
print(f'홈캠/DB 설정 확인 | collection={COLLECTION_NAME} | URL과 인증정보는 출력하지 않음')


홈캠/DB 설정 확인 | collection=face_embeddings | URL과 인증정보는 출력하지 않음


## 2. 등록 얼굴 갤러리와 SCRFD·ArcFace 초기화


In [6]:
import cv2
import numpy as np
import torch

TORCH_DLL_DIR = Path(torch.__file__).resolve().parent / 'lib'
if os.name == 'nt':
    cudnn_dll = TORCH_DLL_DIR / 'cudnn64_9.dll'
    if not cudnn_dll.is_file():
        raise FileNotFoundError(cudnn_dll)
    os.environ['PATH'] = f"{TORCH_DLL_DIR}{os.pathsep}{os.environ.get('PATH', '')}"
    _torch_dll_dir_handle = os.add_dll_directory(str(TORCH_DLL_DIR))
    _cudnn_handle = ctypes.WinDLL(str(cudnn_dll))

import onnxruntime as ort
from insightface.model_zoo import get_model
from pymongo import MongoClient
from pymongo.errors import PyMongoError
from ultralytics import YOLO
from deeplearning.face_identity import FaceGallery, FaceIdentityEngine, GalleryEntry, IdentityThresholds

EXPECTED_METADATA = ('arcface', 'insightface-buffalo_l-w600k_r50-v0.7', 'insightface-norm-crop-112-v1')
student_names: dict[str, str] = {}
gallery_entries: list[GalleryEntry] = []
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=10_000, connectTimeoutMS=10_000)
try:
    client.admin.command('ping')
    projection = {'_id': 0, 'student_id': 1, 'student_name': 1, 'vector': 1, 'dimension': 1, 'normalized': 1, 'model_name': 1, 'model_version': 1, 'preprocessing_version': 1}
    for doc in client[MONGODB_DATABASE][COLLECTION_NAME].find({}, projection):
        student_id = doc.get('student_id')
        metadata = (doc.get('model_name'), doc.get('model_version'), doc.get('preprocessing_version'))
        if not isinstance(student_id, str) or not student_id or doc.get('dimension') != 512 or doc.get('normalized') is not True or metadata != EXPECTED_METADATA:
            raise RuntimeError(f'{student_id!r} 얼굴 벡터 metadata가 현재 ArcFace와 다릅니다.')
        gallery_entries.append(GalleryEntry(student_id, np.asarray(doc.get('vector'), dtype=np.float32)))
        student_names[student_id] = str(doc.get('student_name') or student_id)
except PyMongoError as exc:
    raise RuntimeError('MongoDB 연결/조회에 실패했습니다.') from exc
finally:
    client.close()
if not gallery_entries:
    raise RuntimeError('등록 얼굴 갤러리가 비어 있습니다.')

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
if not torch.cuda.is_available() or 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('PyTorch와 ONNX Runtime CUDA 실행 환경이 필요합니다.')
detector = get_model(str(DETECTOR_PATH), providers=providers)
input_size = int(os.environ.get('FACE_DETECTION_INPUT_SIZE', '1280'))
detector.prepare(ctx_id=0, input_size=(input_size, input_size), det_thresh=float(os.environ.get('FACE_DETECTION_THRESHOLD', '0.4')))
recognizer = get_model(str(RECOGNIZER_PATH), providers=providers)
recognizer.prepare(ctx_id=0)
threshold_data = json.loads(THRESHOLD_FILE.read_text(encoding='utf-8'))
engine = FaceIdentityEngine(
    detector=detector, recognizer=recognizer, gallery=FaceGallery.from_entries(gallery_entries),
    thresholds=IdentityThresholds(float(threshold_data['similarity_threshold']), float(threshold_data['margin_threshold'])),
    detection_threshold=float(os.environ.get('FACE_DETECTION_THRESHOLD', '0.4')),
    identity_min_detection_confidence=float(os.environ.get('FACE_IDENTITY_MIN_DETECTION_CONFIDENCE', '0.6')),
    minimum_face_size=int(os.environ.get('FACE_MINIMUM_SIZE', '40')), preferred_face_size=int(os.environ.get('FACE_PREFERRED_SIZE', '112')),
    minimum_blur_score=float(os.environ.get('FACE_MINIMUM_BLUR_SCORE', '20')), preferred_blur_score=float(os.environ.get('FACE_PREFERRED_BLUR_SCORE', '100')),
    uncertain_quality_threshold=float(os.environ.get('FACE_UNCERTAIN_QUALITY_THRESHOLD', '0.45')),
    use_flip_tta=os.environ.get('FACE_USE_FLIP_TTA', 'true').lower() == 'true',
    tta_similarity_band=float(os.environ.get('FACE_TTA_SIMILARITY_BAND', '0.08')), tta_margin_band=float(os.environ.get('FACE_TTA_MARGIN_BAND', '0.06')),
)
person_model = YOLO(PERSON_MODEL_PATH)
person_model.to('cuda')
detector.detect(np.zeros((input_size, input_size, 3), dtype=np.uint8), max_num=0)
recognizer.get_feat(np.zeros((112, 112, 3), dtype=np.uint8))
print(f'등록 학생 {len(gallery_entries)}명 | SCRFD·ArcFace·YOLO CUDA 준비 완료')


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'enable_cudnn': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_com

## 3. 최신 프레임 수신과 실시간 실행

창에서 `q`를 누르면 종료합니다. ByteTrack ID는 홈캠 안에서만 유효합니다.


In [7]:
from deeplearning.homecam_tracking import (HomecamDiagnostics, PersonTrack, PersonTrackIdentityStore, TrackIdentityStatus, associate_faces_to_people)

class LatestFrameReader:
    def __init__(self, source: str | int, *, is_rtsp: bool) -> None:
        backend = cv2.CAP_FFMPEG if is_rtsp else (cv2.CAP_DSHOW if os.name == 'nt' else cv2.CAP_ANY)
        self._capture = cv2.VideoCapture(source, backend)
        if is_rtsp:
            self._capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        if not self._capture.isOpened():
            self._capture.release()
            source_name = '홈캠 RTSP' if is_rtsp else f'웹캠 {source}'
            raise RuntimeError(f'{source_name} 입력을 열지 못했습니다.')
        self._lock = threading.Lock()
        self._frame = None
        self._captured_at = 0.0
        self._stopped = False
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def _run(self) -> None:
        while not self._stopped:
            ok, frame = self._capture.read()
            if not ok:
                time.sleep(0.02)
                continue
            with self._lock:
                self._frame = frame
                self._captured_at = time.perf_counter()

    def read(self) -> tuple[np.ndarray | None, float | None]:
        with self._lock:
            if self._frame is None:
                return None, None
            return self._frame.copy(), self._captured_at

    def close(self) -> None:
        self._stopped = True
        self._capture.release()
        self._thread.join(timeout=2.0)

def extract_people(result: Any) -> tuple[PersonTrack, ...]:
    boxes = result.boxes
    if boxes is None or boxes.id is None:
        return ()
    xyxy = boxes.xyxy.detach().cpu().numpy()
    track_ids = boxes.id.detach().cpu().numpy().astype(int)
    confidences = boxes.conf.detach().cpu().numpy()
    return tuple(PersonTrack(int(track_id), tuple(int(value) for value in bbox), float(confidence)) for bbox, track_id, confidence in zip(xyxy, track_ids, confidences))

def status_style(status: TrackIdentityStatus) -> tuple[str, tuple[int, int, int]]:
    return {
        TrackIdentityStatus.REGISTERED: ('등록 학생', (0, 200, 0)),
        TrackIdentityStatus.UNKNOWN: ('미등록', (0, 0, 255)),
        TrackIdentityStatus.UNCERTAIN: ('판정 보류', (0, 200, 255)),
    }[status]

def run_homecam_demo() -> None:
    is_rtsp = HOME_CAM_SOURCE == 'rtsp'
    camera_source = HOME_CAM_RTSP_URL if is_rtsp else HOME_CAM_CAMERA_INDEX
    reader = LatestFrameReader(camera_source, is_rtsp=is_rtsp)
    identity_store = PersonTrackIdentityStore(
        history_size=int(os.environ.get('HOME_CAM_IDENTITY_HISTORY_SIZE', '12')),
        minimum_observations=int(os.environ.get('HOME_CAM_IDENTITY_MINIMUM_OBSERVATIONS', '4')),
        stale_frames=int(os.environ.get('HOME_CAM_TRACK_STALE_FRAMES', '30')),
    )
    diagnostics = HomecamDiagnostics()
    source_label = '홈캠 RTSP' if is_rtsp else f'노트북 웹캠 {HOME_CAM_CAMERA_INDEX}'
    window_name = f'{source_label} 얼굴 식별 + ByteTrack v1'
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO)
    frame_index = 0
    faces = ()
    try:
        while True:
            started = time.perf_counter()
            frame, captured_at = reader.read()
            if frame is None:
                time.sleep(0.01)
                continue
            frame_index += 1
            result = person_model.track(frame, persist=True, tracker=TRACKER_CONFIG, classes=[0], conf=PERSON_CONFIDENCE, device='cuda', verbose=False)[0]
            people = extract_people(result)
            recognition_due = frame_index == 1 or frame_index % FACE_INTERVAL == 0
            if recognition_due:
                faces = engine.identify(frame, extract_embeddings=True)
                associations = associate_faces_to_people(people, faces, minimum_face_coverage=FACE_COVERAGE_THRESHOLD)
            else:
                faces = ()
                associations = ()
            identities = identity_store.update(people, faces, associations)
            for identity in identities:
                status_label, color = status_style(identity.status)
                name = student_names.get(identity.student_id, identity.student_id or status_label)
                left, top, right, bottom = identity.bbox
                cv2.rectangle(frame, (left, top), (right, bottom), color, 3)
                cv2.putText(frame, f'P{identity.track_id} {name} [{status_label}] s={identity.similarity:.3f} m={identity.margin:.3f} n={identity.observation_count}', (left, min(frame.shape[0] - 10, bottom + 22)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
            duration = time.perf_counter() - started
            capture_latency = started - captured_at if captured_at is not None else None
            diagnostics.record(people=people, faces=faces, associations=associations, frame_duration=duration, capture_latency=capture_latency)
            cv2.putText(frame, f'FPS {1.0 / max(duration, 1e-6):.1f} | people {len(people)} | q: quit', (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
            cv2.imshow(window_name, frame)
            if cv2.waitKeyEx(1) & 0xFF == ord('q'):
                break
    finally:
        reader.close()
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        DIAGNOSTIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_path = DIAGNOSTIC_OUTPUT_DIR / f"homecam-tracking-v1-{time.strftime('%Y%m%d-%H%M%S')}.json"
        output_path.write_text(json.dumps(diagnostics.snapshot(identity_store), ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'홈캠 추적 진단 결과 저장: {output_path}')

run_homecam_demo()


Exception in thread Thread-9 (_run):
Traceback (most recent call last):
  File "c:\Users\qkddn\anaconda3\envs\smart_monitoring\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\qkddn\anaconda3\envs\smart_monitoring\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\qkddn\AppData\Local\Temp\ipykernel_17568\521579915.py", line 22, in _run
cv2.error: Unknown C++ exception from OpenCV code


홈캠 추적 진단 결과 저장: C:\smart_office_monitoring\deeplearning\training\runs\homecam_tracking\homecam-tracking-v1-20260821-114718.json


## 다음 단계

진단 JSON에서 얼굴-사람 연결률, 신원 변경, 확정 소요시간과 FPS를 먼저 평가합니다. v1이 통과한 뒤에만 홈캠·CCTV 동시 화면과 겹침 구역 측정, 사람 ReID 기반 `global_track_id` 인계를 별도 버전으로 추가합니다.